# Project work part 3 - Data quality

- Author: Mehdi Jamali


## Links
- GitHub: [Click here](https://github.com/Mehdij1511/IND320-Mehdij1511) 

- Streamlit: [Click here](https://ind320mehdij1511.streamlit.app/) 

- Open-Meteo API documentation: [Open-Meteo docs](https://open-meteo.com/en/docs)

- Video: To be added after recording.

## Declaration of AI usage

The following AI tools were used during the development of this project:
- **GitHub Copilot:**
    - Used for code completion, suggestions, and feedback during development.
    - Also used to help implement and test the Part 3 ERA5 downloader, data-quality analyses, Streamlit pages, and caching safeguards.
- **Gemini:**
    - Used for coding assistance and for finding relevant functions and methods.
- **NotebookLM (Gemini):**
    - Used as a research assistant to answer questions and explain topics related to the project.

The suggestions were reviewed, adapted, and tested before they were used in the project.

**The AI tools were used within the guidelines of "K3 - Full use of AI" at NMBU.**

## Screencast

The five-minute screencast will show the Part 3 Streamlit pages, explain how the sampled locations are used for ERA5 downloads, and briefly walk through the SPC, LOF, STL, and spectrogram analyses. Add the video link above after recording.

## Work Log:

I continued Part 3 on the temporary `part3` branch, leaving `main` untouched. I inspected the supplied NVE shapefile: it contains five polygon regions (`RM`, `RN`, `RS`, `RV`, and `RØ`) in ETRS89 / UTM zone 33N. GeoPandas loads the polygons and reproducibly samples 1,000 points per region using a fixed seed, converting coordinates to longitude and latitude. The assignment mapping is NO1=RS+RØ, NO2=RS, NO3=RM, NO4=RM+RN, and NO5=RV. Sample counts and that points lie inside polygons were checked. A separate electricity-area boundary file still needs confirmation.

The downloader requests hourly ERA5 precipitation and 2-metre temperature in UTC. It groups sampled coordinates on 0.25-degree grid cells and weights each cell by its sample count. An oversized 50-location request failed; batches of five work. A one-day live test returned 24 observations and a five-location full-year test returned 8,784. Successful raw batches are atomically cached locally, and complete area/year series are cached after validation. MongoDB can share these caches and quota tracking; no MongoDB secret is configured locally, so SQLite is used. Requests are spaced 110 seconds apart; rolling safeguards are below published API limits, and HTTP 429 is reported without automatic retries. The API reported its daily limit reached and provided a retry delay. Last checked: 78/286 unique NO3 grid cells had full-year data cached; 208 remained. The local usage ledger cannot see all activity on the API account, so the API's quota response takes precedence.

The precipitation analysis high-pass filters with DCT, calculates SATV, estimates robust median/MAD SPC limits and marks outliers on the precipitation plot; SATV itself is not plotted. The temperature analysis uses LOF with a default 1% contamination. Both return plots, summaries and outlier tables. For weekly NO3 filling data from Magasinstatistikk, STL exposes seasonal/trend smoothers and the robust option; the spectrogram exposes window length and overlap. SPC/LOF passed synthetic 8,784-hour tests, and STL/spectrogram ran on live reservoir data. Final weather analyses still require the complete API series.

Streamlit now has a Part 3 home page and two analysis pages: STL/spectrogram tabs and precipitation SPC/temperature LOF tabs. Both use area selectors and the requested controls. Weather downloads require explicit confirmation. AppTest passed, including download-confirmation and quota-warning paths; the local app previously returned HTTP 200. Remaining: resume cached downloads after quota reset; finish all 286 NO3 grid cells; validate 8,784 ordered, non-missing hours; run every notebook cell and inspect plots; export the complete PDF; verify both required NVE geography layers; add the public screencast link and submit. No final PDF exists because the weather series is incomplete.

This work is being developed on the temporary `part3` branch. Changes should remain separate from `main` until peer review and teacher feedback are complete.

## Load and check the NVE region shapefile

The five region codes in the shapefile are kept separate so the sampled coordinates can be traced back to their original polygons. The sampling uses a fixed random seed so it can be reproduced.

## Inspect region names and sample locations

The helper module reads the shapefile, checks the region codes, and creates the reproducible location samples.

In [ ]:
# Import the project helper when the notebook is started from either folder.
%matplotlib inline
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / "part3_data.py").is_file():
    project_root = project_root.parent
if not (project_root / "part3_data.py").is_file():
    raise FileNotFoundError("Could not find part3_data.py in the project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Read the polygons and draw 1,000 reproducible locations in each region.
from part3_data import load_nve_regions, sample_region_locations

regions = load_nve_regions()
region_coordinates = sample_region_locations(
    regions,
    samples_per_region=1000,
    random_seed=42,
)

sample_counts = {code: len(points) for code, points in region_coordinates.items()}
assert set(sample_counts) == {"RM", "RN", "RS", "RV", "RØ"}
assert all(count == 1000 for count in sample_counts.values())

print(regions[["REGIONNAVN"]])
print("Sampled locations per region:", sample_counts)


## Download hourly ERA5 data for NO3

The electricity areas use the corresponding NVE regions: NO1 uses RS and RØ, NO2 uses RS, NO3 uses RM, NO4 uses RM and RN, and NO5 uses RV. This keeps East as RS+RØ and North as RM+RN. The download function uses 1,000 sampled locations per NVE region; locations that fall on the same 0.25-degree ERA5 grid cell are requested once and weighted by their sample count when calculating the hourly regional average. Successfully downloaded API batches and complete weighted area/year series are cached locally so a later run resumes without repeating completed requests.

In [ ]:
from part3_data import download_era5_data

# Download hourly ERA5 precipitation and temperature for all of 2024.
weather_no3_2024 = download_era5_data(
    region_identifier="NO3",
    start_date="2024-01-01",
    end_date="2024-12-31",
    samples_per_region=1000,
    random_seed=42,
)
assert len(weather_no3_2024) == 8784
assert weather_no3_2024["time"].is_monotonic_increasing
assert weather_no3_2024["time"].is_unique
assert weather_no3_2024[["precipitation", "temperature_2m"]].notna().all().all()

# Keep the validated area/year series in the ignored local cache directory.
weather_cache_file = project_root / "data" / "era5_cache" / "NO3_ERA5_2024.csv"
weather_cache_file.parent.mkdir(parents=True, exist_ok=True)
weather_no3_2024.to_csv(weather_cache_file, index=False)

print(f"Downloaded {len(weather_no3_2024):,} hourly observations.")
print(f"Cached complete area series at {weather_cache_file}.")
weather_no3_2024.head()


## Precipitation outliers and temperature anomalies

The precipitation analysis uses a high-pass DCT to calculate SATV and robust SPC limits, while the temperature analysis identifies anomalies with Local Outlier Factor. SATV is used to classify outliers but is not plotted.

In [ ]:
from IPython.display import display
from part3_analysis import plot_precipitation_outliers, plot_temperature_anomalies

# Use the DCT-derived SATV only to classify precipitation SPC outliers.
precipitation_figure, precipitation_summary, precipitation_outliers = (
    plot_precipitation_outliers(
        weather_no3_2024,
        frequency_cutoff=52,
        n_standard_deviations=3,
    )
)
assert precipitation_summary["observations"] == len(weather_no3_2024)
assert len(precipitation_outliers) == precipitation_summary["outliers"]
display(precipitation_figure)
display(pd.Series(precipitation_summary, name="Precipitation SPC summary"))
display(precipitation_outliers.head(10))

# Detect temperature anomalies using Local Outlier Factor at a 1% rate.
temperature_figure, temperature_summary, temperature_anomalies = (
    plot_temperature_anomalies(weather_no3_2024, contamination=0.01)
)
assert temperature_summary["observations"] == len(weather_no3_2024)
assert len(temperature_anomalies) == temperature_summary["anomalies"]
display(temperature_figure)
display(pd.Series(temperature_summary, name="Temperature LOF summary"))
display(temperature_anomalies.head(10))


## Reservoir-filling STL and spectrogram

Weekly filling levels for electricity area 3 (NO3) are retrieved from Magasinstatistikk. STL and the spectrogram use the same time series.

In [ ]:
import matplotlib.pyplot as plt
from nve_data import load_reservoir_data
from part3_analysis import plot_spectrogram, plot_stl_decomposition

# Area number 3 in the EL dataset is the reservoir series for NO3.
reservoir_data = load_reservoir_data()
no3_reservoir_data = reservoir_data.loc[
    (reservoir_data["omrType"] == "EL") & (reservoir_data["omrnr"] == 3)
].sort_values("dato_Id")
no3_filling = no3_reservoir_data.set_index("dato_Id")["fyllingsgrad"]

stl_figure, stl_result = plot_stl_decomposition(
    no3_filling,
    seasonal=13,
    trend=53,
    robust=True,
)
assert len(stl_result.trend) == len(no3_filling)
display(stl_figure)
plt.close(stl_figure)

spectrogram_figure = plot_spectrogram(
    no3_filling,
    window_length=64,
    window_overlap=32,
)
assert spectrogram_figure.axes
display(spectrogram_figure)
plt.close(spectrogram_figure)
